# Russian corpus: TF matrices computing

This notebook loads Russian texts from `data/Russian/`, performs lemmatization with **pymorphy3**
(without removing stop words), and builds term-frequency matrices and JSON files in the same format
as the other datasets: `TfMatrixRU.json`, `authorsRU.json`.

**Requirements:** `pip install pymorphy3 pymorphy3-dicts-ru`

**Note:** the term-frequency matrices used in the experiments are **precomputed** and shipped in `data/`. Re-running this notebook is therefore optional and only required if you want to rebuild the matrices from the raw texts.

In [11]:
# Imports
import os
import re
import json
import numpy as np
import pandas as pd
import pymorphy3
from collections import Counter
import tqdm

## Text cleaning and lemmatization

Keep only Cyrillic, digits, and basic punctuation. Lemmatize with pymorphy3. We do not remove stop words.

In [12]:
def clear_text_ru(text: str) -> str:
    """Clean Russian text and normalise whitespace.
+
    We retain Cyrillic characters and (optionally) Latin letters to avoid deleting
    foreign insertions and named entities that occur in Russian literary prose.
    """
    text = re.sub(r'[^а-яa-zёА-ЯA-ZЁ]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def lemmatize_doc(text: str, morph: pymorphy3.MorphAnalyzer) -> list:
    """Split text into tokens and return list of lemmas (stop words kept)."""
    tokens = text.split()
    return [morph.parse(t)[0].normal_form for t in tokens if t.strip()]

## Load corpus and build TF matrices (DAR-style layout)

Paths are relative to the notebook directory (run from `appendix/`). Outputs go to `data/` (same folder as TfMatrixDE.json, authorsDE.json).

**Filter:** we only process and write to matrices works whose authors have **more than 3** works in the dataset (authors with ≤3 works are excluded).

In [13]:
# Paths: run notebook from appendix/ so data/Russian and data/ resolve correctly
DATA_DIR = os.path.join('data', 'Russian')
OUT_DIR = 'data'
if not os.path.isdir(DATA_DIR) and os.path.isdir(os.path.join('appendix', 'data', 'Russian')):
    os.chdir('appendix')
    DATA_DIR = os.path.join('data', 'Russian')
    OUT_DIR = 'data'

# Only include authors with 3 and more  works in the dataset (matrix built only for these)
MIN_WORKS_PER_AUTHOR = 3

# List all .txt files and count works per author (from filename: Author.WorkName.Year.txt)
all_filenames = sorted(f for f in os.listdir(DATA_DIR) if f.endswith('.txt'))
author_from_file = [f.replace('.txt', '').split('.')[0] for f in all_filenames]
author_counts_all = Counter(author_from_file)
# Authors to keep: those with 3 and more works
authors_keep = {a for a, cnt in author_counts_all.items() if cnt >= MIN_WORKS_PER_AUTHOR}
filenames = [f for f in all_filenames if f.replace('.txt', '').split('.')[0] in authors_keep]

corpus_ru = []
authors_ru = []
booknames_ru = []

morph = pymorphy3.MorphAnalyzer()

print(f"Filter: authors with >=3 works. Kept {len(filenames)} / {len(all_filenames)} files, {len(authors_keep)} / {len(author_counts_all)} authors.")
print(f"Loading and lemmatizing {len(filenames)} files...")

for filename in tqdm.tqdm(filenames, desc="Files"):
    filepath = os.path.join(DATA_DIR, filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        raw = f.read()
    clean = clear_text_ru(raw)
    lemmas = lemmatize_doc(clean, morph)
    corpus_ru.append(lemmas)
    # Filename format: Author.WorkName.Year.txt
    parts = filename.replace('.txt', '').split('.')
    authors_ru.append(parts[0])
    booknames_ru.append('.'.join(parts))

print(f"Loaded {len(corpus_ru)} documents.")

Filter: authors with >=3 works. Kept 639 / 755 files, 89 / 180 authors.
Loading and lemmatizing 639 files...


Files: 100%|██████████| 639/639 [1:22:34<00:00,  7.75s/it]  

Loaded 639 documents.


In [14]:
# Build vocabulary (all lemmas) and document-term counts
vocab = Counter()
doc_counts = []
for lemmas in corpus_ru:
    c = Counter(lemmas)
    doc_counts.append(c)
    vocab.update(c)

# Feature selection: keep the 20,000 most frequent word types (MFW)
MAX_WORDS = 20_000
# Sort words by total count (descending) for MFW ordering, as in TfMatrixDE
word_order = [w for w, _ in vocab.most_common(MAX_WORDS)]
word2idx = {w: i for i, w in enumerate(word_order)}

# Matrix: rows = words (by total count), columns = documents
# Values = relative frequency in document (count_in_doc / doc_length)
n_words = len(word_order)
n_docs = len(booknames_ru)
rel_freq = np.zeros((n_words, n_docs))
total_count = np.zeros(n_words)

for j, (lemmas, c) in enumerate(zip(corpus_ru, doc_counts)):
    doc_len = len(lemmas)
    if doc_len == 0:
        continue
    for w, cnt in c.items():
        if w in word2idx:
            i = word2idx[w]
            rel_freq[i, j] = cnt / doc_len
            total_count[i] += cnt

# DataFrame in TfMatrixDE style: Count, Word, then one column per document
df_ru = pd.DataFrame(
    {'Count': total_count, 'Word': word_order},
    columns=['Count', 'Word'] + booknames_ru
)
df_ru[booknames_ru] = rel_freq

print(f"Vocabulary: {n_words} types (lemmas), {n_docs} documents.")

Vocabulary: 20000 types (lemmas), 639 documents.


In [15]:
# Summary table: total number of works and authors (after filter: authors with >3 works)
n_works = len(corpus_ru)
n_authors = len(set(authors_ru))
summary = pd.DataFrame(
    {
        'Number of works': [n_works],
        'Number of authors': [n_authors],
    }
)
# print("Total (after filter):")
# print(summary.to_string(index=False))
# Before/after filter
summary_full = pd.DataFrame(
    {
        'Stage': ['Before filter (all files)', 'After filter (authors with >3 works)'],
        'Number of works': [len(all_filenames), n_works],
        'Number of authors': [len(author_counts_all), n_authors],
    }
)
# print("\nBefore vs after filter:")
# print(summary_full.to_string(index=False))
display(summary)
display(summary_full)

,Number of works,Number of authors
0,639,89


,Stage,Number of works,Number of authors
0,Before filter (all files),755,180
1,After filter (authors with >3 works),639,89


In [16]:
# Save TfMatrixRU.json (same format as TfMatrixDE.json)
os.makedirs(OUT_DIR, exist_ok=True)
df_ru.to_json(os.path.join(OUT_DIR, 'TfMatrixRU.json'))

# Save authorsRU.json: document index -> author name (same format as authorsDE.json)
authors_series = pd.Series(authors_ru)
authors_series.to_json(os.path.join(OUT_DIR, 'authorsRU.json'), orient='index')

# Save goldRU.json: list of author names in document order (like goldDE.json)
with open(os.path.join(OUT_DIR, 'goldRU.json'), 'w', encoding='utf-8') as f:
    json.dump(authors_ru, f, ensure_ascii=False, indent=4)

print(f"Saved: {OUT_DIR}/TfMatrixRU.json, {OUT_DIR}/authorsRU.json, {OUT_DIR}/goldRU.json")

Saved: data/TfMatrixRU.json, data/authorsRU.json, data/goldRU.json


## A single TF matrix for 3–5-grams

Using the same corpus, we construct **one** relative-frequency matrix whose features jointly comprise lemma 3-grams, 4-grams, and 5-grams (a single shared feature space). The result is saved as `TfMatrixRU_3-5gram.json`.

In [ ]:
def doc_to_ngrams(lemmas: list, n: int, sep: str = "_") -> list:
    """Sliding window: convert a lemma sequence into string n-grams."""
    if len(lemmas) < n:
        return []
    return [sep.join(lemmas[i : i + n]) for i in range(len(lemmas) - n + 1)]


# A single shared vocabulary: all 3-, 4-, and 5-grams combined
ngram_vocab = Counter()
# Per document: a joint ngram in count map for n ∈ {3,4,5}
ngram_doc_counts = []

for lemmas in corpus_ru:
    combined = Counter()
    for n in [3, 4, 5]:
        ngrams = doc_to_ngrams(lemmas, n)
        combined.update(ngrams)
    ngram_doc_counts.append(combined)
    ngram_vocab.update(combined)

MAX_NGRAMS = 20_000  # keep top features by corpus frequency
ngram_order = [g for g, _ in ngram_vocab.most_common(MAX_NGRAMS)]
ngram2idx = {g: i for i, g in enumerate(ngram_order)}

n_terms = len(ngram_order)
rel_freq_ng = np.zeros((n_terms, n_docs))
total_count_ng = np.zeros(n_terms)

for j, c in enumerate(ngram_doc_counts):
    # Document length = total number of n-grams (3+4+5) observed in this document
    doc_ngram_total = sum(c.values())
    if doc_ngram_total == 0:
        continue
    for g, cnt in c.items():
        if g in ngram2idx:
            i = ngram2idx[g]
            rel_freq_ng[i, j] = cnt / doc_ngram_total
            total_count_ng[i] += cnt

df_ng = pd.DataFrame(
    {"Count": total_count_ng, "Word": ngram_order},
    columns=["Count", "Word"] + booknames_ru,
)
df_ng[booknames_ru] = rel_freq_ng

out_name = os.path.join(OUT_DIR, "TfMatrixRU_3-5gram.json")
df_ng.to_json(out_name)
print(f"Saved: {out_name} — {n_terms} features (3+4+5-grams), {n_docs} documents.")

Saved: data\TfMatrixRU_3-5gram.json — 20,000 features (3+4+5-grams), 639 documents.
